# Husky 와이어태핑을 활용한 능동적 전압 오류주입 공격 (Active Wiretapping for Voltage FA)

## 두 ChipWhisperer 장치의 역할 분리 실험 — Lite (정상 호스트) + Husky (능동적 공격자)

---

### 🎯 노트북의 목표

이 노트북은 기존의 **클럭 오류주입(Clock Glitch)** 시나리오를 고도화하여, **전압 오류주입 공격(Voltage Fault Injection)** 환경을 구축하는 방법을 다룹니다. 

실제 공격 시나리오와 유사하게, **CW-Lite는 대상 기기(Target)와 정상적으로 통신하는 호스트 역할**만 수행하며 자신이 공격받고 있다는 사실을 모릅니다. 반면 **Husky는 두 기기 사이의 통신/클럭 라인을 와이어태핑**하여 타이밍을 동기화하고, 타겟의 전원(VCC/VDD) 라인에 물리적으로 개입하여 **치명적인 전압 강하(Voltage Drop)를 주입하는 공격자 역할**을 수행합니다.

### 🔌 ⚡ 하드웨어 결선 가이드 (전압 오류주입 필수 설정 - 매우 중요!) ⚡

전압 오류주입은 클럭 선의 논리 레벨만 바꾸는 클럭 글리치와 달리, 타겟 칩으로 들어가는 **전원 라인을 순간적으로 접지(GND)로 끌어내려야(Crowbar 방식)** 하므로 물리적 배선과 보드 설정이 공격의 성공 여부를 결정합니다.

#### 1. 정상 통신 및 전원 공급 (Lite ↔ Target)
* 기존과 동일하게 **20-pin 리본 케이블**을 Lite와 Target(예: CW308 보드) 사이에 연결합니다. 이 케이블을 통해 전원, 기준 클럭, 트리거, Tx/Rx가 대상 기기에 공급됩니다.

#### 2. 와이어태핑 및 동기화 라인 (Husky ↔ Target)
* **Clock Sync (클럭 훔치기):** Target 보드로 들어가는 Clock 핀을 점퍼선으로 연결하여 Husky의 `HS1` (또는 Target IO IN) 핀에 연결합니다. Husky는 이 클럭을 보고 내부 PLL을 동기화합니다.
* **Trigger Sync (통신 시작점 훔치기):** Target 보드의 Trigger 핀(TIO4)을 점퍼선으로 연결하여 Husky의 `HS2` 핀에 연결합니다.

#### 3. 💥 전압 글리치 타격 라인 (Husky ↔ Target) 💥
* **SMA 동축 케이블 연결:** Husky 전면 패널의 **`Glitch Out` SMA 포트**와 Target 보드(CW308)의 **`GLITCH` SMA 포트**를 SMA 케이블로 직접 연결합니다.
* **Target 보드 하드웨어 점퍼(Jumper) 세팅 (핵심):**
  * 일반적인 상태에서 타겟 칩의 전원 라인에는 안정적인 전원 공급을 위해 **디커플링 커패시터(Decoupling Capacitor)**가 붙어 있습니다. 이 커패시터가 글리치를 방해(필터링)합니다.
  * CW308 UFO 보드의 경우, 칩으로 들어가는 VDD 라인의 커패시터를 우회하거나 분리하기 위해 특정 점퍼(예: `FILT` 핀 주변 점퍼 또는 VDD/Shunt 점퍼)를 제거하거나 이동해야 합니다. (사용하는 타겟 보드의 매뉴얼에서 *Voltage Glitching Setup*을 반드시 확인하세요.)
  * VDD 라인에 직렬로 연결된 Shunt 저항 뒷단에서 글리치가 발생하도록 셋업하여 전압이 확실하게 0V에 가깝게 떨어지도록 구성해야 합니다.

---

| 단계 | 내용 | 핵심 산출물 |
|:----:|:----|:----|
| **1단계** | 다중 장치(Lite + Husky) 동시 연결 및 역할 부여 | `lite_scope`, `husky_scope` |
| **2단계** | 정상 호스트(Lite) ↔ 타겟 통신 채널 확보 | `target` 객체, 펌웨어 플래싱 |
| **3단계** | 공격자(Husky)의 신호 동기화 및 ⚡전압 글리치⚡ 모듈 설정 | `husky_scope.glitch` (enable_only) 설정 완료 |
| **4단계** | 광범위 파라미터 탐색 (Voltage Fault Injection Campaign) | 3중 루프 실행, `cglitch_result` |
| **5단계** | 공격 결과 시각화 및 최적 파라미터 도출 | Bokeh 산점도 시각화 |


In [ ]:
import chipwhisperer as cw
import time
import numpy as np
from tqdm.notebook import trange

# Bokeh 시각화 도구
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.palettes import Category10
import scipy.stats as sp
output_notebook()

### 1단계: 장치 식별 및 동시 연결

Lite와 Husky를 동시에 제어하기 위해 Serial Number(SN)를 사용하여 명시적으로 연결합니다.
`cw.list_devices()`를 실행하여 각 장치의 SN을 확인하고 아래 코드에 입력하세요.

In [ ]:
# 현재 연결된 기기 목록 확인 (주석 해제 후 실행)
# print(cw.list_devices())

# TODO: 각 연구자의 환경에 맞게 SN을 수정하세요.
SN_LITE = "442031204630..."  # CW-Lite (Host)
SN_HUSKY = "502031203841..." # CW-Husky (Attacker)

try:
    if not lite_scope.connectStatus:
        pass
except NameError:
    print("[Lite] 정상 호스트 장치 연결 중...")
    lite_scope = cw.scope(sn=SN_LITE)
    
try:
    if not husky_scope.connectStatus:
        pass
except NameError:
    print("[Husky] 공격자 장치 연결 중...")
    husky_scope = cw.scope(sn=SN_HUSKY)

print("✅ 두 장치 모두 연결 완료!")

### 2단계: 정상 호스트(Lite) 설정 및 타겟 프로그래밍

CW-Lite는 Target 보드에 클럭을 공급하고 통신을 담당합니다. 타겟 보드를 초기화하고 간단한 루프 연산을 수행하는 펌웨어를 업로드합니다.

In [ ]:
# Lite를 통신 컨트롤러로 설정
target = cw.target(lite_scope, cw.targets.SimpleSerial2)

# Lite 클럭 및 통신 설정 (표준 설정)
lite_scope.default_setup()

print("🔧 타겟 펌웨어 컴파일 및 업로드 (Lite 활용)...")
cw.program_target(lite_scope, cw.programmers.STM32FProgrammer, "../hardware/victims/firmware/simpleserial-glitch/simpleserial-glitch-CW308_STM32F3.hex")

time.sleep(0.5)
print("✅ 타겟 통신/클럭 설정 완료 (Lite)")

### 3단계: 공격자(Husky) 와이어태핑 동기화 및 ⚡전압 글리치⚡ 설정

Husky는 내부 클럭을 사용하지 않고, Target으로 들어가는 Lite의 클럭 신호를 도청하여 동기화합니다. 
**[주의]** 클럭 글리치(`clock_xor`)와 달리, 전압 글리치를 위해서는 **Husky 내부의 MOSFET을 구동하여 SMA 포트의 전압을 GND로 떨어뜨리는 `enable_only` 모드**를 사용해야 합니다.

In [ ]:
# 3.1 클럭 와이어태핑 (Husky 동기화)
# Husky의 Target IO IN 핀으로 들어오는 외부 클럭을 소스로 사용합니다.
husky_scope.clock.clkgen_src = 'extclk_aux_io'
time.sleep(0.5)

# 주파수 확인 및 PLL 동기화
ext_freq = husky_scope.clock.extclk_freq
print(f"[Husky] 감지된 외부 클럭 주파수: {ext_freq/1e6:.2f} MHz")

husky_scope.clock.clkgen_freq = ext_freq # 발견한 클럭으로 주파수 락(Lock)
husky_scope.clock.adc_mul = 1 # 1 sample = 1 clock

# 3.2 트리거 와이어태핑
# Target이 통신을 시작할 때 발생하는 Trigger 신호(tio4)를 가로챕니다.
husky_scope.trigger.triggers = 'tio4'

# 3.3 전압 글리치 모듈 설정 (Voltage Glitch)
husky_scope.glitch.clk_src = 'pll'           # 동기화된 클럭 기반

# ⚡ 핵심 변경: 'enable_only' 모드는 설정된 width 동안 Crowbar 트랜지스터를 활성화하여
# Glitch SMA 포트와 연결된 타겟의 VCC를 GND로 단락(쇼트)시킵니다.
husky_scope.glitch.output = 'enable_only'    

husky_scope.glitch.trigger_src = 'ext_single' # 가로챈 트리거 1회에 반응

print("✅ Husky 와이어태핑 및 전압 글리치(Voltage Glitch) 모듈 준비 완료!")

### 4단계: 오류주입 파라미터 탐색 (Voltage Fault Injection Campaign)

전압 글리치는 회로 내부의 저항 및 커패시턴스 성분으로 인해 클럭 글리치보다 전압이 떨어지는 데 약간의 시간이 더 필요합니다. 따라서 대체로 **`width` 값이 클럭 글리치보다 커야 유효한 타격이 들어가는 경향**이 있습니다.
타겟 조작은 Lite가, 공격 준비(Arm)는 Husky가 담당합니다.

In [ ]:
# 타겟을 안전하게 재부팅하기 위한 헬퍼 함수 (Lite를 통해 전원/리셋 제어)
def reboot_target():
    lite_scope.io.nrst = 'low'
    time.sleep(0.05)
    lite_scope.io.nrst = 'high_z'
    time.sleep(0.05)

# 정상 응답값 확인 (Lite를 통해 요청)
reboot_target()
target.simpleserial_write('g', bytearray([]))
expected_ret = target.simpleserial_read('r', 4)
print(f"🎯 정상 동작 시 반환값 (expected_ret): {expected_ret}")

# 탐색 범위 설정 (전압 글리치 특성에 맞게 조정)
# 주의: 타겟의 전원단 상태에 따라 필요 Width가 크게 다릅니다. 
# 반응이 아예 없다면 widths를 더 크게(예: 10~150), 칩이 계속 죽기만 한다면 작게 줄이세요.
ext_offsets = range(10, 30, 2)  # 글리치 시점 (클럭 사이클 딜레이)
offsets = np.arange(-40, 40, 5) # 클럭 내부 미세 위상
widths = np.arange(10, 80, 5)   # 글리치 폭 (전압 강하 유지 시간, 보통 클럭글리치보다 넓음)

results = []

print("🚀 전압 오류주입 캠페인 시작...")
for ext_offset in trange(len(ext_offsets), desc="Ext Offset 탐색"):
    husky_scope.glitch.ext_offset = ext_offsets[ext_offset]
    
    for offset in offsets:
        husky_scope.glitch.offset = offset
        
        for width in widths:
            husky_scope.glitch.width = width
            
            # 1. 대상 통신 버퍼 초기화
            target.flush()
            
            # 2. 공격자(Husky) 장전 - 트리거를 기다림
            husky_scope.arm()
            
            # 3. 호스트(Lite)가 정상 명령 전송 -> 이 순간 Trigger가 발생하며 Husky가 전압 강하 주입
            target.simpleserial_write('g', bytearray([]))
            
            # 4. 결과 읽기 및 분류
            ret = target.simpleserial_read('r', 4)
            
            if ret is None: # 응답 없음 (전압 강하가 너무 커서 칩이 다운됨)
                status = "reset"
                reboot_target()
            elif ret == expected_ret: # 정상 응답 (전압 강하가 부족하여 공격 실패)
                status = "normal"
            else: # 비정상 응답 (전압 오류 주입 성공!!)
                status = "success"
                print(f"🎉 SUCCESS at Ext_off:{husky_scope.glitch.ext_offset}, Off:{offset}, Wid:{width} | Ret: {ret}")
                
            # 결과 저장
            results.append((husky_scope.glitch.ext_offset, offset, width, status))

print("✅ 전압 오류주입 캠페인 완료!")

### 5단계: 공격 결과 시각화 (Bokeh)

수집된 결과를 바탕으로, 어느 `(offset, width)` 파라미터 조합에서 성공적인 공격(Success)과 시스템 리셋(Reset)이 발생하는지 분포를 확인합니다. 전압 글리치의 경우 주로 Width 축을 따라 성공과 리셋 구간이 뚜렷하게 나뉘는 경향을 볼 수 있습니다.

In [ ]:
# 결과 데이터 정리
data_normal = {'x': [], 'y': []}
data_success = {'x': [], 'y': []}
data_reset = {'x': [], 'y': []}

for ext_off, off, wid, status in results:
    if status == 'normal':
        data_normal['x'].append(wid)
        data_normal['y'].append(off)
    elif status == 'success':
        data_success['x'].append(wid)
        data_success['y'].append(off)
    elif status == 'reset':
        data_reset['x'].append(wid)
        data_reset['y'].append(off)

# Bokeh 플롯 생성
p = figure(title="Voltage Fault Injection Results (Husky Wiretapping)", x_axis_label="Glitch Width (Voltage Drop Duration)", y_axis_label="Glitch Offset")

p.scatter(data_normal['x'], data_normal['y'], color="green", legend_label="Normal", alpha=0.1, size=5)
p.scatter(data_reset['x'], data_reset['y'], color="red", legend_label="Reset (Crash)", alpha=0.5, size=5)
p.scatter(data_success['x'], data_success['y'], color="blue", legend_label="Success (Fault!)", marker="star", size=12)

p.legend.click_policy = "hide"
show(p)

### 📝 요약 및 핵심 개념

| 구분 | 설명 |
|:---|:---|
| **물리적 배선** | 전압 글리치는 **반드시 SMA 동축 케이블 연결** 및 보드의 **커패시터(Bypass/Decoupling) 제어 세팅**이 선행되어야 동작합니다. |
| **출력 모드** | Husky의 글리치 출력 모드를 `enable_only`로 설정하여 타겟 전원 라인을 직접 제어(Crowbar)하도록 변경했습니다. |
| **Width 민감도** | 전압 글리치는 클럭 글리치에 비해 Width 파라미터 변화에 더 민감하고 대체로 더 넓은 펄스폭을 요구합니다. |

실험이 끝난 후에는 장치를 안전하게 해제합니다.

In [ ]:
try:
    lite_scope.dis()
    husky_scope.dis()
    target.dis()
    print("✅ 모든 장치 안전하게 해제 완료.")
except:
    pass